Se importan las librerias y herramientas necesarias para la actividad.

In [17]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

Se carga el dataset.

In [18]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


Se divide el dataset en proporción: 80% train, 10% validation y 10% test.

In [19]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

Se define el device y se hace uso de seed para obtener resultados reproducibles.

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Using device: cuda


Se establecen los límites del clipping se obtienen los valores de mean y std.

In [21]:
lat_lon_cols = ['Latitude', 'Longitude']
processed_cols = [c for c in x_train.columns if c not in lat_lon_cols]

Q1 = x_train[processed_cols].quantile(0.25)
Q3 = x_train[processed_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = torch.tensor((Q1 - 1.5 * IQR).values, dtype=torch.float32)
upper_bound = torch.tensor((Q3 + 1.5 * IQR).values, dtype=torch.float32)
mean = torch.tensor(x_train[processed_cols].mean().values, dtype=torch.float32)
std = torch.tensor(x_train[processed_cols].std().values, dtype=torch.float32)

Se define la capa para realizar el Standard Scaling en los datos.

In [22]:
class ScalingLayer(nn.Module):
    def __init__(self, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / (self.std + 1e-8)

Separa el tensor en dos grupos para poder aplicar el preprocesamiento a todas las características menos a Latitude y Longitude.

In [23]:
class PreprocessingLayer(nn.Module):
    processed_cols_idx = [0, 1, 2, 3, 4, 5]
    latlon_idx = [6, 7]

    def __init__(self, mean, std, lower, upper):
        super().__init__()
        self.scaler = ScalingLayer(mean, std)
        self.register_buffer('lower', lower)
        self.register_buffer('upper', upper)

    def forward(self, x):                                      
        x_main = x[:, self.processed_cols_idx]
        x_latlon = x[:, self.latlon_idx]
        x_main = torch.clamp(x_main, self.lower, self.upper) 
        x_main = self.scaler(x_main)                         
        return torch.cat([x_main, x_latlon], dim=1)

In [24]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [25]:
train_data = HousingDataset(x_train.values, y_train.values)
val_data = HousingDataset(x_val.values,   y_val.values)
test_data = HousingDataset(x_test.values,  y_test.values)

Se define la arquitectura del modelo, las capas ocultas y funciones de activación.

In [26]:
class HousingNetwork(nn.Module):
    def __init__(self, hidden1_size, hidden2_size, hidden3_size, mean, std, lower, upper):
        super().__init__()
        self.preprocessing = PreprocessingLayer(mean, std, lower, upper)

        self.network = nn.Sequential(
            nn.Linear(8, hidden1_size),   
            nn.ReLU(),
            nn.Linear(hidden1_size, hidden2_size),
            nn.ReLU(),
            nn.Linear(hidden2_size, hidden3_size),
            nn.ReLU(),
            nn.Linear(hidden3_size, 1)   
        )

    def forward(self, x):
        x = self.preprocessing(x)
        return self.network(x).squeeze(1)

La función de entrenamiento que se ha utilizado en actividades pasadas, al igual que el early stopping.

In [27]:
def train(_model, _train_loader, _val_loader, _criterion, _optimizer, _num_epochs):
    train_losses = []
    val_losses   = []
    patience = 5
    min_delta = 1e-4
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_weights = None

    for epoch in range(_num_epochs):
        _model.train()
        running_loss = 0.0

        for X_batch, y_batch in tqdm(_train_loader, desc=f"Epoch {epoch+1}/{_num_epochs}"):
            _optimizer.zero_grad()
            outputs = _model(X_batch)
            loss = _criterion(outputs, y_batch)
            loss.backward()
            _optimizer.step()
            running_loss += loss.item() * X_batch.size(0)

        epoch_train_loss = running_loss / len(_train_loader.dataset)

        _model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in _val_loader:
                val_outputs = _model(X_val)
                val_loss += _criterion(val_outputs, y_val).item() * X_val.size(0)

        epoch_val_loss = val_loss / len(_val_loader.dataset)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

        if epoch_val_loss < best_val_loss - min_delta:
            best_val_loss = epoch_val_loss
            best_weights = _model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                _model.load_state_dict(best_weights)
                break

    return train_losses, val_losses

In [28]:
criterion = nn.MSELoss()

Se crean 3 modelos con distintos hiperparámetros.

In [29]:
# Modelo 1. 128-64-32 Adam
model = HousingNetwork(
    hidden1_size=128, hidden2_size=64, hidden3_size=32,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses, val_losses = train(model, train_loader, val_loader, criterion, optimizer, num_epochs)

Epoch 1/50: 100%|██████████| 258/258 [00:00<00:00, 287.52it/s]


Epoch 1 | Train Loss: 1.1343 | Val Loss: 0.6773


Epoch 2/50: 100%|██████████| 258/258 [00:00<00:00, 272.44it/s]


Epoch 2 | Train Loss: 0.6514 | Val Loss: 0.5895


Epoch 3/50: 100%|██████████| 258/258 [00:00<00:00, 274.70it/s]


...

Epoch 48 | Train Loss: 0.4206 | Val Loss: 0.5034


Epoch 49/50: 100%|██████████| 258/258 [00:00<00:00, 331.79it/s]


Epoch 49 | Train Loss: 0.4217 | Val Loss: 0.4344


Epoch 50/50: 100%|██████████| 258/258 [00:00<00:00, 335.46it/s]

Epoch 50 | Train Loss: 0.4177 | Val Loss: 0.4449
Early stopping at epoch 50


In [30]:
# Modelo 2: 64-64-32 Adam
model2 = HousingNetwork(
    hidden1_size=64, hidden2_size=64, hidden3_size=32,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)

optimizer = optim.Adam(model2.parameters(), lr=0.001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses2, val_losses2 = train(model2, train_loader, val_loader, criterion, optimizer, num_epochs)

Epoch 1/50: 100%|██████████| 258/258 [00:00<00:00, 304.45it/s]


Epoch 1 | Train Loss: 1.3960 | Val Loss: 0.9999


Epoch 2/50: 100%|██████████| 258/258 [00:00<00:00, 311.35it/s]


Epoch 2 | Train Loss: 0.7579 | Val Loss: 0.6565


Epoch 3/50: 100%|██████████| 258/258 [00:00<00:00, 306.90it/s]


...

Epoch 14 | Train Loss: 0.4683 | Val Loss: 0.5333


Epoch 15/50: 100%|██████████| 258/258 [00:00<00:00, 325.51it/s]


Epoch 15 | Train Loss: 0.4777 | Val Loss: 0.4991


Epoch 16/50: 100%|██████████| 258/258 [00:00<00:00, 329.42it/s]


Epoch 16 | Train Loss: 0.4669 | Val Loss: 0.4934
Early stopping at epoch 16


In [31]:
# Modelo 3: 256-128-64 SGD
model3 = HousingNetwork(
    hidden1_size=256, hidden2_size=128, hidden3_size=64,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)
    
optimizer = optim.SGD(model3.parameters(), lr=0.0001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses, val_losses = train(model3, train_loader, val_loader, criterion, optimizer, num_epochs)

Epoch 1/50: 100%|██████████| 258/258 [00:00<00:00, 337.15it/s]


Epoch 1 | Train Loss: 1.4132 | Val Loss: 1.2932


Epoch 2/50: 100%|██████████| 258/258 [00:00<00:00, 342.49it/s]


Epoch 2 | Train Loss: 1.3276 | Val Loss: 1.2911


Epoch 3/50: 100%|██████████| 258/258 [00:00<00:00, 341.35it/s]


...

Epoch 44 | Train Loss: 0.9148 | Val Loss: 0.8246


Epoch 45/50: 100%|██████████| 258/258 [00:00<00:00, 369.44it/s]


Epoch 45 | Train Loss: 0.8789 | Val Loss: 1.0530


Epoch 46/50: 100%|██████████| 258/258 [00:00<00:00, 371.95it/s]


Epoch 46 | Train Loss: 0.9059 | Val Loss: 0.8133
Early stopping at epoch 46


In [32]:
test_loader = DataLoader(test_data, batch_size=64)
results = {}

for name, m in [("Modelo 1", model), ("Modelo 2", model2), ("Modelo 3", model3)]:
    m.eval()
    test_loss_total = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            test_outputs = m(X_batch)
            test_loss_total += criterion(test_outputs, y_batch).item() * X_batch.size(0)

    test_mse = test_loss_total / len(test_loader.dataset)
    results[name] = test_mse
    print(f"{name} | Test MSE: {test_mse:.4f} | Test RMSE: {test_mse**0.5:.4f}")

best_model_name = min(results, key=results.get)
print(f"\nMejor modelo: {best_model_name} con RMSE: {results[best_model_name]**0.5:.4f}")

Modelo 1 | Test MSE: 0.4426 | Test RMSE: 0.6653
Modelo 2 | Test MSE: 0.5074 | Test RMSE: 0.7123
Modelo 3 | Test MSE: 0.7960 | Test RMSE: 0.8922

Mejor modelo: Modelo 1 con RMSE: 0.6653


Al comparar los modelos se puede apreciar que el modelo con mejor rendimiento es el primero, seguido por el segundo y finalmente el tercero en cuanto a rendimiento se refiere. En esta implementación se hizo uso de clipping para el manejo de los outliers, debido a que la acción de eliminarlos dentro del nn.Module causaría que se rompa el tamaño del batch. Así que la implementación más adecuada que permita seguir utilizando el preprocessing dentro del nn.Module es el clipping.